<a href="https://colab.research.google.com/github/andrewbeyou88/uXUbYJ34sxDN11/blob/main/microwakeword/notebooks/basic_training_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

In the comment at the start of certain blocks, I note some specific settings to consider modifying.

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.

In [2]:
# Installs microWakeWord. Be sure to restart the session after this is finished.
import platform

if platform.system() == "Darwin":
    # `pymicro-features` is installed from a fork to support building on macOS
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# `audio-metadata` is installed from a fork to unpin `attrs` from a version that breaks Jupyter
!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

!git clone https://github.com/kahrendt/microWakeWord
!pip install -e ./microWakeWord

  Cloning https://github.com/whatsnowplaying/audio-metadata (to revision d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f) to /tmp/pip-req-build-8f3d0_w9
  Running command git clone --filter=blob:none --quiet https://github.com/whatsnowplaying/audio-metadata /tmp/pip-req-build-8f3d0_w9
  Running command git rev-parse -q --verify 'sha^d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
  Running command git fetch -q https://github.com/whatsnowplaying/audio-metadata d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Running command git checkout -q d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Resolved https://github.com/whatsnowplaying/audio-metadata to commit d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.0/82.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!pip install piper-sample-generator
!mkdir -p voices
!wget -O voices/en_US-lessac-medium.onnx 'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx?download=true'
!wget -O voices/en_US-lessac-medium.onnx.json 'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx.json?download=true'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 15.9 MB/s eta 0:00:00
  Created wheel for webrtcvad: filename=webrtcvad-2.0.10-cp312-cp312-linux_x86_64.whl size=73523 sha256=8d3536cd92dbbafc05179715b789f9f4eb8637a3a7ad538eb24af37b38f7be77
  Stored in directory: /root/.cache/pip/wheels/1e/d3/95/680fa3b16848f1a58d2edaed34c496224c89a9bc63e17b3614
Successfully built webrtcvad
  Attempting uninstall: librosa
    Found existing installation: librosa 0.11.0
    Uninstalling librosa-0.11.0:
      Successfully uninstalled librosa-0.11.0
  Attempting uninstall: audiomentations
    Found existing installation: audiomentations 0.43.1
   

In [4]:
!mkdir -p /usr/local/lib/python3.12/dist-packages/piper_train/vits
!touch /usr/local/lib/python3.12/dist-packages/piper_train/__init__.py
!touch /usr/local/lib/python3.12/dist-packages/piper_train/vits/__init__.py
!touch /usr/local/lib/python3.12/dist-packages/piper_train/vits/commons.py

In [5]:
# Generates 1 sample of the target word for manual verification.

target_word = 'hey_beemo'  # Phonetic spellings may produce better samples

import os
import sys
import platform

from IPython.display import Audio

if not os.path.exists("./piper-sample-generator"):
    if platform.system() == "Darwin":
        !git clone -b mps-support https://github.com/kahrendt/piper-sample-generator
    else:
        !git clone https://github.com/rhasspy/piper-sample-generator

    !wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

    # Install system dependencies
    !pip install torch torchaudio piper-phonemize-cross==1.2.1

    if "piper-sample-generator/" not in sys.path:
        sys.path.append("piper-sample-generator/")

!python3 -m piper_sample_generator 'hey_beemo' \
--model voices/en_US-lessac-medium.onnx \
--max-samples 1 \
--output-dir generated_samples

Audio("generated_samples/0.wav", autoplay=True)

DEBUG:__main__:Loading ['voices/en_US-lessac-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: voices/en_US-lessac-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)


In [6]:
# Generates a larger amount of wake word samples.
# Start here when trying to improve your model.
# See https://github.com/rhasspy/piper-sample-generator for the full set of
# parameters. In particular, experiment with noise-scales and noise-scale-ws,
# generating negative samples similar to the wake word, and generating many more
# wake word samples, possibly with different phonetic pronunciations.

!python3 -m piper_sample_generator 'hey beemo' \
--model voices/en_US-lessac-medium.onnx \
--max-samples 1000 \
--output-dir generated_samples

DEBUG:__main__:Loading ['voices/en_US-lessac-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: voices/en_US-lessac-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)


In [12]:
import os
import shutil
import scipy.io.wavfile
import scipy.signal
import numpy as np
from pathlib import Path
from tqdm import tqdm
import soundfile as sf

# 1. Curățare folder vechi
output_dir = "./audioset_16k"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

print("Downloading ambient noise archive...")

# 2. Descărcare directă a arhivei de zgomote
!wget -q --show-progress -O speech_commands.tar.gz http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz
!mkdir -p speech_commands_raw

# Extracting all .wav files from background noise directory
!tar -xf speech_commands.tar.gz -C speech_commands_raw --wildcards '*_background_noise_/*.wav'

# 3. Procesare și salvare în ./audioset_16k
bg_files = list(Path("speech_commands_raw").glob("**/*.wav"))
print(f"Gasite {len(bg_files)} fisiere de zgomot ambiental.")

for i, file_path in enumerate(tqdm(bg_files, desc="Processing Ambient Noise")):
    try:
        data, samplerate = sf.read(str(file_path))
        if len(data.shape) > 1:
            data = data.mean(axis=1) # Conversie la mono

        if samplerate != 16000:
            num_samples = int(len(data) * 16000 / samplerate)
            data = scipy.signal.resample(data, num_samples)

        scipy.io.wavfile.write(os.path.join(output_dir, f"{i}.wav"), 16000, (data * 32767).astype(np.int16))
    except Exception as e:
        print(f"Eroare la {file_path}: {e}")

# Curățare fișiere temporare
!rm -rf speech_commands_raw speech_commands.tar.gz

print("Gata! Zgomotul de fundal a fost generat cu succes.")

speech_commands.tar 100%[===================>]   2.26G  22.0MB/s    in 1m 50s  
Gasite 6 fisiere de zgomot ambiental.


Processing Ambient Noise: 100%|██████████| 6/6 [00:00<00:00, 116.67it/s]


Gata! Zgomotul de fundal a fost generat cu succes.


In [16]:
import sys
import os

# Verificăm dacă folderul proiectului există în mediul curent
if os.path.exists("micro-wake-word-models"):
    sys.path.append(os.path.abspath("micro-wake-word-models"))
    print("Folderul micro-wake-word-models a fost găsit și adăugat la cale!")
elif os.path.exists("micro_wake_word"):
    sys.path.append(os.path.abspath("."))
    print("Modulul local a fost găsit!")
else:
    print("Clonăm depozitul oficial...")
    !git clone https://github.com/esphome/micro-wake-word-models.git
    sys.path.append(os.path.abspath("micro-wake-word-models"))

Clonăm depozitul oficial...
Cloning into 'micro-wake-word-models'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 98 (delta 46), reused 33 (delta 33), pack-reused 43 (from 1)
Receiving objects: 100% (98/98), 629.04 KiB | 407.00 KiB/s, done.
Resolving deltas: 100% (52/52), done.


In [19]:
import sys
import os

# Instalăm doar pachetele valide de augmentare audio
!pip install -q audiomentations

# Adăugăm calea locală
repo_path = os.path.abspath("micro-wake-word-models")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("Mediul este pregătit!")

Mediul este pregătit!


In [23]:
import os
import scipy.io.wavfile
import numpy as np
import datasets
from tqdm import tqdm

os.makedirs("mit_rirs", exist_ok=True)
print("Populăm folderul mit_rirs...")

# Incarcam dataset-ul FARA decodare audio automata (decode=False)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train")
rir_dataset = rir_dataset.cast_column("audio", datasets.Audio(decode=False))

for i, row in enumerate(tqdm(rir_dataset, desc="Saving RIRs")):
    audio_info = row['audio']

    # Daca avem cale catre fisier local, il citim direct
    if 'path' in audio_info and audio_info['path'] and os.path.exists(audio_info['path']):
        with open(audio_info['path'], 'rb') as f_in, open(f"mit_rirs/{i}.wav", 'wb') as f_out:
            f_out.write(f_in.read())
    # Daca avem octeti (bytes) salvali direct
    elif 'bytes' in audio_info and audio_info['bytes']:
        with open(f"mit_rirs/{i}.wav", 'wb') as f_out:
            f_out.write(audio_info['bytes'])

print(f"Gata! S-au salvat {len(os.listdir('mit_rirs'))} fișiere RIR în mit_rirs.")

Populăm folderul mit_rirs...


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

Saving RIRs: 100%|██████████| 270/270 [00:00<00:00, 4353.33it/s]

Gata! S-au salvat 270 fișiere RIR în mit_rirs.


In [26]:
import audiomentations

# Adăugăm alias pentru AddColorNoise dacă nu există direct în modul
if not hasattr(audiomentations, "AddColorNoise"):
    if hasattr(audiomentations, "AddGaussianNoise"):
        audiomentations.AddColorNoise = audiomentations.AddGaussianNoise
    elif hasattr(audiomentations.augmentations, "add_color_noise"):
        audiomentations.AddColorNoise = audiomentations.augmentations.add_color_noise.AddColorNoise

print("Patch-ul pentru audiomentations a fost aplicat cu succes!")

Patch-ul pentru audiomentations a fost aplicat cu succes!


In [28]:
import audiomentations
from audiomentations import AddGaussianNoise

class FixedColorNoise(AddGaussianNoise):
    def __init__(self, min_snr_db=-10, max_snr_db=30, p=0.5, **kwargs):
        # Convertim SNR în limite de amplitudine aproximativ echivalente pentru AddGaussianNoise
        super().__init__(min_amplitude=0.001, max_amplitude=0.015, p=p)

# Suprascriem referința din audiomentations
audiomentations.AddColorNoise = FixedColorNoise

print("Patch-ul pentru AddColorNoise a fost aplicat cu succes!")

Patch-ul pentru AddColorNoise a fost aplicat cu succes!


In [30]:
# Sets up the augmentations.
# To improve your model, experiment with these settings and use more sources of
# background clips.

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities = {
                                "SevenBandParametricEQ": 0.1,
                                "TanhDistortion": 0.1,
                                "PitchShift": 0.1,
                                "BandStopFilter": 0.1,
                                "AddColorNoise": 0.1,
                                "AddBackgroundNoise": 0.75,
                                "Gain": 1.0,
                                "RIR": 0.5,
                            },
                         impulse_paths = ['mit_rirs'],
                         background_paths = ['fma_16k', 'audioset_16k'],
                         background_min_snr_db = -5,
                         background_max_snr_db = 10,
                         min_jitter_s = 0.195,
                         max_jitter_s = 0.205,
                         )


In [ ]:
# Augment a random clip and play it back to verify it works well

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

Audio("augmented_clip.wav", autoplay=True)

In [31]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'generated_augmented_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=10,    # Uses the same spectrogram repeatedly, just shifted over by one frame. This simulates the streaming inferences while training/validating in nonstreaming mode.
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=1,    # The testing set uses the streaming version of the model, so no artificial repetition is necessary
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [32]:
# Downloads pre-generated spectrogram features (made for microWakeWord in
# particular) for various negative datasets. This can be slow!

output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        link = link_root + fname

        zip_path = f"negative_datasets/{fname}"
        !wget -O {zip_path} {link}
        !unzip -q {zip_path} -d {output_dir}

--2026-08-16 14:07:16--  https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/dinner_party.zip
Resolving huggingface.co (huggingface.co)... 13.35.202.34, 13.35.202.40, 13.35.202.97, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.34|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/65e327bc1445a768ed343b8c/228d7e72cd5fdc4e6e57da36b88a4c227d34cb8dc44041078b4c4b65dc75848d?response-content-type=application%2Fzip&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27dinner_party.zip%3B+filename%3D%22dinner_party.zip%22%3B&X-Xet-Cas-Uid=public&user_id=public&Expires=1786892836&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjVlMzI3YmMxNDQ1YTc2OGVkMzQzYjhjLzIyOGQ3ZTcyY2Q1ZmRjNGU2ZTU3ZGEzNmI4OGE0YzIyN2QzNGNiOGRjNDQwNDEwNzhiNGM0YjY1ZGM3NTg0OGRcXD9yZXNwb25zZS1jb250ZW50LXR5cGU9YXBwbGljYXRpb24lMkZ6aXAmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj1pbmxpbmU

In [33]:
# Save a yaml config that controls the training process
# These hyperparamters can make a huge different in model quality.
# Experiment with sampling and penalty weights and increasing the number of
# training steps.

import yaml
import os

config = {}

config["window_step_ms"] = 10

config["train_dir"] = (
    "trained_models/wakeword"
)


# Each feature_dir should have at least one of the following folders with this structure:
#  training/
#    ragged_mmap_folders_ending_in_mmap
#  testing/
#    ragged_mmap_folders_ending_in_mmap
#  testing_ambient/
#    ragged_mmap_folders_ending_in_mmap
#  validation/
#    ragged_mmap_folders_ending_in_mmap
#  validation_ambient/
#    ragged_mmap_folders_ending_in_mmap
#
#  sampling_weight: Weight for choosing a spectrogram from this set in the batch
#  penalty_weight: Penalizing weight for incorrect predictions from this set
#  truth: Boolean whether this set has positive samples or negative samples
#  truncation_strategy = If spectrograms in the set are longer than necessary for training, how are they truncated
#       - random: choose a random portion of the entire spectrogram - useful for long negative samples
#       - truncate_start: remove the start of the spectrogram
#       - truncate_end: remove the end of the spectrogram
#       - split: Split the longer spectrogram into separate spectrograms offset by 100 ms. Only for ambient sets

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    { # Only used for validation and testing
        "features_dir": "negative_datasets/dinner_party_eval",
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]

# Number of training steps in each iteration - various other settings are configured as lists that corresponds to different steps
config["training_steps"] = [10000]

# Penalizing weight for incorrect class predictions - lists that correspond to training steps
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]

config["learning_rates"] = [
    0.001,
]  # Learning rates for Adam optimizer - list that corresponds to training steps
config["batch_size"] = 128

config["time_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["time_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["freq_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps

config["eval_step_interval"] = (
    500  # Test the validation sets after every this many steps
)
config["clip_duration_ms"] = (
    1500  # Maximum length of wake word that the streaming model will accept
)

# The best model weights are chosen first by minimizing the specified minimization metric below the specified target_minimization
# Once the target has been met, it chooses the maximum of the maximization metric. Set 'minimization_metric' to None to only maximize
# Available metrics:
#   - "loss" - cross entropy error on validation set
#   - "accuracy" - accuracy of validation set
#   - "recall" - recall of validation set
#   - "precision" - precision of validation set
#   - "false_positive_rate" - false positive rate of validation set
#   - "false_negative_rate" - false negative rate of validation set
#   - "ambient_false_positives" - count of false positives from the split validation_ambient set
#   - "ambient_false_positives_per_hour" - estimated number of false positives per hour on the split validation_ambient set
config["target_minimization"] = 0.9
config["minimization_metric"] = None  # Set to None to disable

config["maximization_metric"] = "average_viable_recall"

with open(os.path.join("training_parameters.yaml"), "w") as file:
    documents = yaml.dump(config, file)

In [52]:
import yaml
import os

config_path = "/content/trained_models/wakeword/training_config.yaml"

if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = yaml.unsafe_load(f)

    # 1. Menținem batch_size redus
    config["batch_size"] = 64

    # 2. Eliminăm complet dinner_party_eval din lista de features
    original_features = config.get("features", [])
    config["features"] = [
        f for f in original_features
        if "dinner_party_eval" not in f.get("features_dir", "")
    ]

    with open(config_path, "w") as f:
        yaml.dump(config, f)

    print("Setul de evaluare greu a fost eliminat cu succes!")
else:
    print("Fișierul nu a fost găsit.")

Setul de evaluare greu a fost eliminat cu succes!


In [53]:
# Trains a model. When finished, it will quantize and convert the model to a
# streaming version suitable for on-device detection.
# It will resume if stopped, but it will start over at the configured training
# steps in the yaml file.
# Change --train 0 to only convert and test the best-weighted model.
# On Google colab, it doesn't print the mini-batch results, so it may appear
# stuck for several minutes! Additionally, it is very slow compared to training
# on a local GPU.

!python -m microwakeword.model_train_eval \
--training_config='/content/trained_models/wakeword/training_config.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

INFO:absl:Loading and analyzing data sets.
2026-08-16 15:04:24.897258: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1786892664.898788   25635 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (64, 204, 40)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (64, 204, 1, 4